## Cell 1 — Load labeled + filter unlabeled pool

Load the 100 labeled users from Step 3/5, load the full candidate pool (946 rows), and filter to the unlabeled subset (846 users) by removing usernames already labeled.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths
folder    = Path.cwd()                                # Classification/
DATA_DIR  = folder.parent / 'data'                    # Iran_POI/data/
ITER1_DIR = folder / 'Iteration_1' / 'Step5_Analysis'
ITER2_DIR = folder / 'Iteration_2'
ITER2_DIR.mkdir(exist_ok=True)

# 1) Labeled pool from Step 3 (with English translations added in Step 5)
labeled = pd.read_csv(ITER1_DIR / 'iteration_1_consensus_translated.csv')
print(f"Labeled users:        {len(labeled)}")

# 2) Full candidate pool — the source the 100 labeled users were drawn from
pool = pd.read_csv(DATA_DIR / 'Candidates_user_data_MERGED.csv')
print(f"Full candidate pool:  {len(pool)}")

# 3) Filter to unlabeled: pool minus rows whose username is in labeled
labeled_keys = set(labeled['username'].astype(str).str.lower().str.strip())
unlabeled = pool[~pool['username'].astype(str).str.lower().str.strip().isin(labeled_keys)].copy()
unlabeled = unlabeled.reset_index(drop=True)
print(f"Unlabeled users:      {len(unlabeled)}")

# 4) Sanity check — labeled should overlap fully with pool
overlap = labeled['username'].astype(str).str.lower().isin(
    pool['username'].astype(str).str.lower()
).sum()
print(f"\nSanity check: {overlap}/{len(labeled)} labeled users found in the pool (should be 100/100)")

Labeled users:        100
Full candidate pool:  946
Unlabeled users:      846

Sanity check: 100/100 labeled users found in the pool (should be 100/100)


## Cell 2 — Build 11 numeric features (no translation)

Compute the same 11 features as Step 5, but with one change: `bio_mentions_iran` / `name_mentions_iran` / `location_mentions_iran` use a **multilingual** keyword list (English + Persian + Arabic) on **raw text** — no Google Translate.

Same function runs on BOTH `labeled` and `unlabeled` so the feature definitions are identical for training and predicting.

In [4]:
# Multilingual Iran keyword list — covers the 3 main languages of our data.
iran_keywords_multilingual = [
    # English
    'iran', 'iranian', 'persian', 'persia', 'tehran', 'shiraz', 'esfahan',
    'isfahan', 'mashhad', 'tabriz', 'kerman', 'qom', 'farsi',
    # Persian / Farsi
    'ایران', 'ایرانی', 'تهران', 'شیراز', 'اصفهان', 'مشهد', 'تبریز', 'فارسی',
    # Arabic
    'إيران', 'ايران', 'إيراني', 'ايراني', 'طهران', 'شيراز',
]


def has_iran(text):
    """Return 1 if the text contains any Iran-related keyword (in any of 3 languages)."""
    if pd.isna(text):
        return 0
    s = str(text).lower()
    return int(any(kw in s for kw in iran_keywords_multilingual))


def build_numeric(df):
    """Compute the 11 numeric features used by the Step 5 winner."""
    df = df.copy()

    # Make sure count columns are numeric, fill NaN with 0
    for c in ['followers_count', 'following_count', 'statuses_count']:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

    # Bio length — count chars of RAW description (language-agnostic)
    df['bio_length'] = df['description'].fillna('').astype(str).str.len()

    # Followers / following ratio (+1 avoids divide-by-zero)
    df['followers_following_ratio'] = df['followers_count'] / (df['following_count'] + 1)

    # Account age in years from created_at ("May 2024" format)
    df['created_at_dt'] = pd.to_datetime(df['created_at'], format='%B %Y', errors='coerce')
    df['account_age_years'] = ((pd.Timestamp.today() - df['created_at_dt']).dt.days / 365.25)

    # Binary flags
    df['has_description'] = df['description'].notna().astype(int)
    df['has_location']    = df['location'].notna().astype(int)

    # Iran keyword flags — on RAW text using the multilingual list
    df['bio_mentions_iran']      = df['description'].apply(has_iran)
    df['name_mentions_iran']     = df['display_name'].apply(has_iran)
    df['location_mentions_iran'] = df['location'].apply(has_iran)

    return df


# Run the same function on both datasets
labeled   = build_numeric(labeled)
unlabeled = build_numeric(unlabeled)

# Fill any remaining NaN account_age_years (users with missing created_at) with the labeled median
median_age = labeled['account_age_years'].median()
labeled['account_age_years']   = labeled['account_age_years'].fillna(median_age)
unlabeled['account_age_years'] = unlabeled['account_age_years'].fillna(median_age)

# The 11 feature names — same list for both
numerical_features = [
    'followers_count', 'following_count', 'statuses_count',
    'followers_following_ratio', 'bio_length', 'account_age_years',
    'has_description', 'has_location',
    'bio_mentions_iran', 'name_mentions_iran', 'location_mentions_iran',
]

# Sanity check
print(f"Numeric features built: {len(numerical_features)}\n")
print(f"  labeled shape:   {labeled[numerical_features].shape}")
print(f"  unlabeled shape: {unlabeled[numerical_features].shape}\n")

print("Iran keyword hit rates (sanity check):")
for c in ['bio_mentions_iran', 'name_mentions_iran', 'location_mentions_iran']:
    print(f"  {c:<25s} labeled={labeled[c].sum():>3d}/{len(labeled)}   "
          f"unlabeled={unlabeled[c].sum():>3d}/{len(unlabeled)}")

Numeric features built: 11

  labeled shape:   (100, 11)
  unlabeled shape: (846, 11)

Iran keyword hit rates (sanity check):
  bio_mentions_iran         labeled=  7/100   unlabeled= 59/846
  name_mentions_iran        labeled=  1/100   unlabeled=  9/846
  location_mentions_iran    labeled=  9/100   unlabeled= 75/846


## Cell 3 — Train the Step 5 winner, predict on unlabeled

Re-train LogReg + numeric (your Step 5 winner for `target_population`) on the full 100 labeled users with the new multilingual feature definitions. Then apply it to the 846 unlabeled users to get probabilities and uncertainty scores.

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1) Scale the 11 numeric features. Fit on labeled, transform both.
scaler = StandardScaler()
X_labeled   = scaler.fit_transform(labeled[numerical_features].values)
X_unlabeled = scaler.transform(unlabeled[numerical_features].values)

# 2) Train LogReg (Step 5 winner) on ALL 100 labeled users for target_population.
#    No cross-validation here — we use every labeled example because we want the
#    best possible model for predicting on the unlabeled pool.
y_labeled = labeled['target_population'].values
model = LogisticRegression(max_iter=2000, class_weight=None, random_state=42)
model.fit(X_labeled, y_labeled)

print(f"Trained on {len(y_labeled)} labeled rows.")
print(f"Classes the model can predict: {list(model.classes_)}")
print(f"Sanity-check training accuracy: {model.score(X_labeled, y_labeled):.3f}")

# 3) Predict on unlabeled — get probabilities for each class.
proba = model.predict_proba(X_unlabeled)
predicted_class = model.predict(X_unlabeled)

# Map the model's class labels to fixed prob_0 / prob_1 / prob_2 columns.
# (model.classes_ might be [0, 1, 2] but could be shorter if a class was missing.)
proba_full = np.zeros((proba.shape[0], 3))
for j, c in enumerate(model.classes_):
    proba_full[:, int(c)] = proba[:, j]

# 4) Compute confidence_level (max prob) and uncertainty_score (1 - confidence).
confidence_level  = proba_full.max(axis=1)
uncertainty_score = 1.0 - confidence_level

# 5) Quick sanity: predicted class distribution + uncertainty stats
print(f"\nPredicted class distribution on {len(unlabeled)} unlabeled users:")
for c in [0, 1, 2]:
    count = int((predicted_class == c).sum())
    print(f"  class {c}: {count:>3d} ({100*count/len(unlabeled):>4.1f}%)")

print(f"\nUncertainty score stats:")
print(f"  min:    {uncertainty_score.min():.3f}")
print(f"  median: {np.median(uncertainty_score):.3f}")
print(f"  max:    {uncertainty_score.max():.3f}")

Trained on 100 labeled rows.
Classes the model can predict: [np.int64(0), np.int64(1), np.int64(2)]
Sanity-check training accuracy: 0.720

Predicted class distribution on 846 unlabeled users:
  class 0: 410 (48.5%)
  class 1:  78 ( 9.2%)
  class 2: 358 (42.3%)

Uncertainty score stats:
  min:    0.000
  median: 0.245
  max:    0.640


## Cell 4 — Save the predictions CSV + top-100 to-label file

Builds the PDF-page-14 columns into one DataFrame, sorts by uncertainty (most uncertain first), saves:
- `iteration_2_unlabeled_users_predictions.csv` — full 846 rows
- `iteration_2_users_to_label.csv` — top 100 most uncertain, with empty label columns ready for manual filling

In [ ]:
# 1) Build the predictions DataFrame in the order the PDF (page 14) expects.
predictions = unlabeled[[
    'username', 'display_name', 'description', 'location',
    'followers_count', 'following_count', 'statuses_count', 'created_at',
]].copy()

# Add a clickable Twitter URL right after username for the manual labelers.
predictions.insert(1, 'profile_url', 'https://x.com/' + predictions['username'].astype(str))

predictions['predicted_class']    = predicted_class
predictions['confidence_level']   = confidence_level.round(4)
predictions['prob_0']             = proba_full[:, 0].round(4)
predictions['prob_1']             = proba_full[:, 1].round(4)
predictions['prob_2']             = proba_full[:, 2].round(4)
predictions['uncertainty_score']  = uncertainty_score.round(4)

# 2) Sort by uncertainty descending (most uncertain user is row 0).
predictions = predictions.sort_values('uncertainty_score', ascending=False).reset_index(drop=True)

# 3) Save the full predictions CSV.
predictions_path = ITER2_DIR / 'iteration_2_unlabeled_users_predictions.csv'
predictions.to_csv(predictions_path, index=False)
print(f"Saved {len(predictions)} rows to: {predictions_path.relative_to(folder)}")

# 4) Build the "to label" file — top 100 most uncertain + empty label columns.
#    SAFETY GUARD: if the file already exists (you may have already labeled it!),
#    don't overwrite. This prevents losing your manual labels on re-runs.
to_label_path = ITER2_DIR / 'iteration_2_users_to_label.csv'

if to_label_path.exists():
    print(f"\n⚠ SKIPPING — {to_label_path.name} already exists.")
    print("   If you want to regenerate it (and LOSE your manual labels), delete the file first.")
else:
    to_label = predictions.head(100).copy()
    to_label['target_population']      = ''
    to_label['locals_vs_diaspora']     = ''
    to_label['person_vs_organization'] = ''
    to_label['comments']               = ''
    to_label.to_csv(to_label_path, index=False)
    print(f"Saved top-100 to-label file to: {to_label_path.relative_to(folder)}")

# 5) Preview the 5 most uncertain users so you can eyeball that the picks look reasonable.
print("\nTop 5 most uncertain users (your first manual labels):")
print(predictions[[
    'username', 'profile_url', 'predicted_class', 'confidence_level', 'uncertainty_score',
]].head(5).to_string(index=False))

## Cell 5 — Merge the new manual labels with iteration 1 → 200 labeled rows

Load the 100 new labels you just produced, fill any blanks with `2` (unknown), save 3 separate manual_labels CSVs (PDF page 15 compliance), then concatenate with the iteration 1 labeled set to get a 200-row combined dataset for retraining.

In [7]:
# 1) Load the manual labels you just produced.
manual = pd.read_csv(ITER2_DIR / 'iteration_2_users_to_label.csv')
print(f"Loaded {len(manual)} manually-labeled users.")

# 2) Fill any blank/NaN label cells with 2 ('unknown'). Convert to int.
for col in ['target_population', 'locals_vs_diaspora', 'person_vs_organization']:
    manual[col] = pd.to_numeric(manual[col], errors='coerce').fillna(2).astype(int)

print(f"\nClass distributions in your new 100 labels:")
for col in ['target_population', 'locals_vs_diaspora', 'person_vs_organization']:
    print(f"  {col}: {dict(manual[col].value_counts().sort_index())}")

# 3) Save 3 separate manual-labels CSVs per the PDF page 15 spec.
common_cols = ['username', 'display_name', 'description', 'location',
                'followers_count', 'following_count', 'statuses_count', 'created_at']

for col in ['target_population', 'locals_vs_diaspora', 'person_vs_organization']:
    out_path = ITER2_DIR / f'iteration_2_manual_labels_{col}.csv'
    manual[common_cols + [col, 'comments']].to_csv(out_path, index=False)
    print(f"Saved: {out_path.relative_to(folder)}")

# 4) Load iteration 1 labeled set and keep only the columns we need.
iter1_path = ITER1_DIR / 'iteration_1_consensus_translated.csv'
iter1 = pd.read_csv(iter1_path)
iter1_slim = iter1[common_cols + ['target_population', 'locals_vs_diaspora', 'person_vs_organization']].copy()
iter1_slim['iteration_added'] = 1

# 5) Build the iteration 2 slim slice from manual.
iter2_slim = manual[common_cols + ['target_population', 'locals_vs_diaspora', 'person_vs_organization']].copy()
iter2_slim['iteration_added'] = 2

# 6) Concatenate → 200 rows.
combined = pd.concat([iter1_slim, iter2_slim], ignore_index=True)
print(f"\nCombined dataset: {len(combined)} rows (= 100 + 100)")
print(f"\nClass distributions in the combined 200-row set:")
for col in ['target_population', 'locals_vs_diaspora', 'person_vs_organization']:
    print(f"  {col}: {dict(combined[col].value_counts().sort_index())}")

# 7) Save the combined dataset for Cell 6 to load.
combined_path = ITER2_DIR / 'iteration_2_combined_labeled.csv'
combined.to_csv(combined_path, index=False)
print(f"\nSaved combined 200-row dataset to: {combined_path.relative_to(folder)}")

Loaded 100 manually-labeled users.

Class distributions in your new 100 labels:
  target_population: {2: np.int64(100)}
  locals_vs_diaspora: {2: np.int64(100)}
  person_vs_organization: {2: np.int64(100)}
Saved: Iteration_2/iteration_2_manual_labels_target_population.csv
Saved: Iteration_2/iteration_2_manual_labels_locals_vs_diaspora.csv
Saved: Iteration_2/iteration_2_manual_labels_person_vs_organization.csv

Combined dataset: 200 rows (= 100 + 100)

Class distributions in the combined 200-row set:
  target_population: {0: np.int64(50), 1: np.int64(13), 2: np.int64(137)}
  locals_vs_diaspora: {0: np.int64(2), 1: np.int64(6), 2: np.int64(192)}
  person_vs_organization: {0: np.int64(19), 1: np.int64(48), 2: np.int64(133)}

Saved combined 200-row dataset to: Iteration_2/iteration_2_combined_labeled.csv


## Cell 6 — Build features for the 200-row combined set

This is Step 5's feature-prep done again for 200 rows instead of 100:
1. Translate the new 100 users (the iter 1 hundred already have translations cached).
2. Build the 11 numeric features (multilingual keywords on raw text, same as Cell 2).
3. Build the 7 TF-IDF feature sets on translated text.
4. Stack into a 9-feature-set dictionary `feature_sets_2` that Cell 7's loop will iterate over.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
import scipy.sparse as sp

# 1) Load the combined 200-row labeled dataset (saved by Cell 5)
combined = pd.read_csv(ITER2_DIR / 'iteration_2_combined_labeled.csv')
print(f"Loaded combined dataset: {len(combined)} rows")

# 2) Add English translations. iter 1 rows already have them; only iter 2 need new translation.
iter1_translated = pd.read_csv(ITER1_DIR / 'iteration_1_consensus_translated.csv')
iter1_translations = iter1_translated.set_index(
    iter1_translated['username'].astype(str).str.lower().str.strip()
)[['description_en', 'display_name_en']]

# Initialize translation columns from the iter 1 lookup
combined_key = combined['username'].astype(str).str.lower().str.strip()
combined['description_en']  = combined_key.map(iter1_translations['description_en'])
combined['display_name_en'] = combined_key.map(iter1_translations['display_name_en'])

# Translate any row still missing translations (the new 100 iter 2 users)
need_translation = combined['description_en'].isna() | combined['display_name_en'].isna()
print(f"Rows needing translation: {need_translation.sum()} (the iter 2 new ones)")

translation_cache = ITER2_DIR / 'iteration_2_combined_translated.csv'
if translation_cache.exists():
    cached = pd.read_csv(translation_cache)
    combined = cached
    print("Loaded translations from cache.")
elif need_translation.any():
    from deep_translator import GoogleTranslator
    import time

    def translate_safe(text):
        if pd.isna(text) or str(text).strip() == '':
            return ''
        try:
            return GoogleTranslator(source='auto', target='en').translate(str(text)[:4500])
        except Exception:
            return str(text)

    print("Translating new descriptions...")
    for i in combined.index[need_translation]:
        combined.at[i, 'description_en']  = translate_safe(combined.at[i, 'description'])
        combined.at[i, 'display_name_en'] = translate_safe(combined.at[i, 'display_name'])
        if (i + 1) % 20 == 0:
            print(f"  {i+1}/{len(combined)} done")
        time.sleep(0.1)
    combined.to_csv(translation_cache, index=False)
    print(f"Saved cache: {translation_cache.relative_to(folder)}")

print(f"\nAfter translation step: {len(combined)} rows, {combined['description_en'].notna().sum()} have description_en")

# 3) Build numeric features (uses build_numeric from Cell 2 — multilingual keywords on raw text)
combined = build_numeric(combined)
median_age = combined['account_age_years'].median()
combined['account_age_years'] = combined['account_age_years'].fillna(median_age)

# 4) Build 7 TF-IDF feature sets on translated text (using build_tfidf style from Step 5)
def build_tfidf_combined(columns, min_df=2):
    text = combined[columns[0]].fillna('').astype(str)
    for c in columns[1:]:
        text = text + ' ' + combined[c].fillna('').astype(str)
    vec = TfidfVectorizer(max_features=300, lowercase=True, stop_words='english', min_df=min_df)
    return vec.fit_transform(text), vec

tfidf_sets_2 = {
    'desc':                build_tfidf_combined(['description_en'], min_df=2),
    'username':            build_tfidf_combined(['username'], min_df=1),
    'fullname':            build_tfidf_combined(['display_name_en'], min_df=1),
    'desc_user':           build_tfidf_combined(['description_en', 'username'], min_df=2),
    'desc_fullname':       build_tfidf_combined(['description_en', 'display_name_en'], min_df=2),
    'user_fullname':       build_tfidf_combined(['username', 'display_name_en'], min_df=1),
    'desc_user_fullname':  build_tfidf_combined(['description_en', 'username', 'display_name_en'], min_df=2),
}

# 5) Build the 9 feature sets dict (7 TF-IDF + numeric + desc+numeric)
scaler_2 = StandardScaler(with_mean=False)
numeric_scaled_2 = sp.csr_matrix(scaler_2.fit_transform(combined[numerical_features].values))

feature_sets_2 = {name: matrix for name, (matrix, _vec) in tfidf_sets_2.items()}
feature_sets_2['numeric']      = numeric_scaled_2
feature_sets_2['desc+numeric'] = sp.hstack([tfidf_sets_2['desc'][0], numeric_scaled_2]).tocsr()

print(f"\nReady. {len(feature_sets_2)} feature sets prepared for 200 rows:")
for name, matrix in feature_sets_2.items():
    print(f"  {name:<22s} shape={matrix.shape}")

Loaded combined dataset: 200 rows
Rows needing translation: 127 (the iter 2 new ones)
Loaded translations from cache.

After translation step: 200 rows, 150 have description_en

Ready. 9 feature sets prepared for 200 rows:
  desc                   shape=(200, 149)
  username               shape=(200, 200)
  fullname               shape=(200, 300)
  desc_user              shape=(200, 149)
  desc_fullname          shape=(200, 178)
  user_fullname          shape=(200, 300)
  desc_user_fullname     shape=(200, 178)
  numeric                shape=(200, 11)
  desc+numeric           shape=(200, 160)


## Cell 7 — Re-run the Step 5 experiment loop on the 200-row dataset (~60 min)

Same 1,296 experiments as Step 5, but with `iteration=2` and the bigger labeled set. Saves the CSV every 50 rows so progress isn't lost.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, LeaveOneOut
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import time

try:
    from xgboost import XGBClassifier
    XGBOOST_OK = True
except Exception as e:
    XGBOOST_OK = False
    print(f"XGBoost not available ({type(e).__name__}). Skipping it.")


# --- Helper functions (modified: SVM uses probability=False under LOOCV) ---

def make_model(algo_name, balanced, training_type=None):
    """Create a fresh classifier. For SVM+LOOCV, disable probability=True
    to avoid the killer Platt-scaling slowdown on imbalanced data."""
    cw = 'balanced' if balanced else None
    if algo_name == 'LogReg':
        return LogisticRegression(max_iter=2000, class_weight=cw, random_state=42)
    if algo_name == 'DecisionTree':
        return DecisionTreeClassifier(class_weight=cw, random_state=42)
    if algo_name == 'RandomForest':
        return RandomForestClassifier(n_estimators=100, class_weight=cw, random_state=42, n_jobs=-1)
    if algo_name == 'SVM':
        prob = (training_type != 'LOOCV')  # K-Fold keeps full AUC; LOOCV drops it for speed.
        return SVC(kernel='linear', probability=prob, class_weight=cw, random_state=42)
    if algo_name == 'AdaBoost':
        return AdaBoostClassifier(random_state=42)
    if algo_name == 'XGBoost':
        return XGBClassifier(eval_metric='mlogloss', random_state=42, verbosity=0, n_jobs=-1)
    raise ValueError(f"Unknown algorithm: {algo_name}")


def class_counts(y):
    c = pd.Series(y).value_counts().to_dict()
    return {0: int(c.get(0, 0)), 1: int(c.get(1, 0)), 2: int(c.get(2, 0))}


def sample_weights_for_balance(y):
    y = np.asarray(y)
    classes, counts = np.unique(y, return_counts=True)
    w = {c: len(y) / (len(classes) * cnt) for c, cnt in zip(classes, counts)}
    return np.array([w[v] for v in y])


def run_one_experiment(X, y, algo_name, balanced, training_type, target_column, feature_set_name, n_classes, iteration):
    y = np.asarray(y)
    if training_type == 'K-Fold':
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        K_val = 5
    else:
        cv = LeaveOneOut()
        K_val = ''

    all_true, all_pred = [], []
    proba_chunks = []
    accuracy = precision = recall = f1 = auc = None

    try:
        for train_idx, test_idx in cv.split(np.zeros(len(y)), y):
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]
            if len(np.unique(y_tr)) < 2:
                continue
            model = make_model(algo_name, balanced, training_type=training_type)
            fit_kwargs = {}
            if balanced and algo_name in ('AdaBoost', 'XGBoost'):
                fit_kwargs['sample_weight'] = sample_weights_for_balance(y_tr)
            model.fit(X_tr, y_tr, **fit_kwargs)
            y_pred = model.predict(X_te)
            all_pred.extend(y_pred)
            all_true.extend(y_te)
            if hasattr(model, 'predict_proba'):
                proba_chunks.append((model.classes_, model.predict_proba(X_te), list(test_idx)))

        if not all_pred:
            raise RuntimeError("no predictions made")

        accuracy  = accuracy_score(all_true, all_pred)
        precision = precision_score(all_true, all_pred, average='weighted', zero_division=0)
        recall    = recall_score(all_true, all_pred, average='weighted', zero_division=0)
        f1        = f1_score(all_true, all_pred, average='weighted', zero_division=0)

        if proba_chunks:
            global_classes = sorted(set(y))
            proba_full = np.zeros((len(y), len(global_classes)))
            for classes_seen, block, idxs in proba_chunks:
                col_map = [global_classes.index(c) for c in classes_seen]
                for k, idx in enumerate(idxs):
                    for j, col in enumerate(col_map):
                        proba_full[idx, col] = block[k, j]
            try:
                if len(global_classes) == 2:
                    auc = roc_auc_score(y, proba_full[:, 1])
                else:
                    auc = roc_auc_score(y, proba_full, multi_class='ovr', average='weighted')
            except Exception:
                auc = None
    except Exception:
        pass

    counts = class_counts(y)
    nonzero = [v for k, v in counts.items() if (n_classes == 3 or k != 2) and v > 0]
    min_size = min(nonzero) if nonzero else 0

    return {
        'iteration':       iteration,
        'target_column':   target_column,
        '#classes':        n_classes,
        '#class_0':        counts[0],
        '#class_1':        counts[1],
        '#class_2':        counts[2] if n_classes == 3 else 0,
        'min_class_size':  min_size,
        'training_type':   training_type,
        'K':               K_val,
        'algorithm':       algo_name,
        'feature_set':     feature_set_name,
        'Features_count':  X.shape[1],
        'balanced':        balanced,
        'accuracy':        accuracy,
        'precision':       precision,
        'recall':          recall,
        'F1':              f1,
        'AUC':             auc,
    }


# --- The loop with RESUME logic ---

ALGOS = ['LogReg', 'DecisionTree', 'RandomForest', 'SVM', 'AdaBoost']
if XGBOOST_OK:
    ALGOS.append('XGBoost')

TASKS = [
    ('target_population',      combined['target_population'].values),
    ('locals_vs_diaspora',     combined['locals_vs_diaspora'].values),
    ('person_vs_organization', combined['person_vs_organization'].values),
]

COL_ORDER = [
    'iteration', 'target_column', '#classes', '#class_0', '#class_1', '#class_2',
    'min_class_size', 'training_type', 'K', 'algorithm', 'feature_set',
    'Features_count', 'balanced', 'accuracy', 'precision', 'recall', 'F1', 'AUC',
]

# After the May 2026 reorg, all iter-2 deliverables live inside Iteration_2/.
CSV_PATH = ITER2_DIR / 'experiments_results_iteration_2.csv'

# Load existing partial CSV — start from where we left off.
if CSV_PATH.exists():
    existing = pd.read_csv(CSV_PATH)
    results_2 = existing.to_dict('records')
    completed = set()
    for _, row in existing.iterrows():
        completed.add((
            row['target_column'], int(row['#classes']), row['algorithm'],
            row['feature_set'], row['training_type'], bool(row['balanced']),
        ))
    print(f"Resuming with {len(results_2)} existing rows. Will only run combinations not yet done.")
else:
    results_2 = []
    completed = set()
    print("Starting fresh — no existing CSV.")


def save_progress():
    pd.DataFrame(results_2)[COL_ORDER].to_csv(CSV_PATH, index=False)


t0 = time.time()
new_count = 0

for target_name, y_full in TASKS:
    for n_classes in [3, 2]:
        if n_classes == 2:
            mask = (y_full != 2)
            y = y_full[mask]
        else:
            mask = None
            y = y_full

        if len(np.unique(y)) < 2:
            print(f"  Skipping {target_name} ({n_classes}cls): only one class")
            continue

        for fset_name, X_full in feature_sets_2.items():
            X = X_full[mask] if mask is not None else X_full
            for algo_name in ALGOS:
                for training_type in ['K-Fold', 'LOOCV']:
                    for balanced in [True, False]:
                        key = (target_name, n_classes, algo_name, fset_name, training_type, bool(balanced))
                        if key in completed:
                            continue  # already done in a previous run

                        row = run_one_experiment(
                            X, y, algo_name, balanced, training_type,
                            target_name, fset_name, n_classes, iteration=2,
                        )
                        results_2.append(row)
                        completed.add(key)
                        new_count += 1

                        if new_count % 50 == 0:
                            save_progress()
                            elapsed = time.time() - t0
                            total = len(results_2)
                            print(f"  ... {new_count} new ({total} total)  "
                                  f"{elapsed:.0f}s elapsed  [partial CSV saved]")

save_progress()
print(f"\nDone. Ran {new_count} new experiments in {time.time()-t0:.0f}s. Total rows: {len(results_2)}")
print(f"Saved to: {CSV_PATH}")

## Cell 8 — Iteration comparison plot (PDF page 16 deliverable)

Loads both iteration CSVs, computes mean Accuracy per iteration, draws the trend line (X = iteration, Y = accuracy), saves `plot_iteration_comparison.png`. Also produces a small summary table.

In [ ]:
import matplotlib.pyplot as plt

# 1) Load both iteration CSVs (each now lives in its iteration folder per PDF page 15).
iter1 = pd.read_csv(folder / 'Iteration_1' / 'experiments_results_iteration_1.csv')
iter2 = pd.read_csv(folder / 'Iteration_2' / 'experiments_results_iteration_2.csv')

# 2) Build a small summary table for the PDF page 16 deliverable.
def summary(df, iteration_number, n_labeled):
    return {
        'iteration':            iteration_number,
        'n_labeled_users':      n_labeled,
        'n_experiments':        len(df),
        'mean_accuracy':        round(df['accuracy'].mean(), 4),
        'mean_F1':              round(df['F1'].mean(), 4),
        'mean_AUC':             round(df['AUC'].mean(), 4),
    }

summary_df = pd.DataFrame([
    summary(iter1, 1, 100),
    summary(iter2, 2, 200),
])

summary_df.to_csv(ITER2_DIR / 'iteration_comparison_summary.csv', index=False)
print("Iteration comparison summary:")
print(summary_df.to_string(index=False))

# 3) Build the trend plot — both an overall line and per-task lines for context.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: overall mean accuracy across iterations (PDF page 16 requirement)
axes[0].plot(summary_df['iteration'], summary_df['mean_accuracy'],
              marker='o', linewidth=2, markersize=10, color='steelblue')
for _, r in summary_df.iterrows():
    axes[0].annotate(f"{r['mean_accuracy']:.3f}",
                      (r['iteration'], r['mean_accuracy']),
                      textcoords='offset points', xytext=(10, 5), fontsize=11)
axes[0].set_xticks(summary_df['iteration'])
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Mean Accuracy (across all 1,296 experiments)')
axes[0].set_title('Overall mean accuracy by iteration')
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 1)

# Right: per-task accuracy (3-class only, for clarity)
tasks = ['target_population', 'locals_vs_diaspora', 'person_vs_organization']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for tgt, color in zip(tasks, colors):
    m1 = iter1[(iter1['target_column']==tgt) & (iter1['#classes']==3)]['accuracy'].mean()
    m2 = iter2[(iter2['target_column']==tgt) & (iter2['#classes']==3)]['accuracy'].mean()
    axes[1].plot([1, 2], [m1, m2], marker='o', linewidth=2, markersize=9, label=tgt, color=color)
axes[1].set_xticks([1, 2])
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Mean Accuracy (3-class only)')
axes[1].set_title('Per-task mean accuracy (3-class)')
axes[1].legend(loc='best', fontsize=9)
axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1)

plt.tight_layout()
out_path = ITER2_DIR / 'plot_iteration_comparison.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\nSaved plot to: {out_path.relative_to(folder)}")